# MTH3302 : Méthodes probabilistes et statistiques pour l'I.A.

Jonathan Jalbert<br/>
Professeur agrégé au Département de mathématiques et de génie industriel<br/>
Polytechnique Montréal<br/>


# TD9 : Régression bayésienne

La régression ridge est un cas particulier de la régression linéaire bayésienne lorsque la loi *a priori* partiellement informative suivante est utilisée :

$$ f_{(\boldsymbol{\beta},\sigma^2)}(\boldsymbol{\beta},\sigma^2) \propto \mathcal{N}\left(\boldsymbol{\beta} \, \left| \, \boldsymbol{0}_p , \frac{\sigma^2}{\lambda}I_p \right. \right) \times \frac{1}{\sigma^2}. $$

Pour être conséquent avec l'approche bayésienne, il faudrait fixer l'hyperparamètre $\lambda$ avant même d'avoir vu les données. De façon pragmatique, la valeur de $\lambda$ est souvent choisie par validation comme nous le ferons dans cet exercice.

## Données

Le jeu de données étudiées pour l'exercice 1 provient de l'étude de Tibshirani et al. (2004) illustrant les méthodes de régression modernes telle que la régression ridge. Nous étudierons la glycémie sanguine (glu) en fonction de 8 variables explicatives : l'âge du patient (age), l'indice de masse corporel du patient (bmi), de la tension artérielle moyenne (map) ainsi que de 5 autres indicateur sanguins (tc, ldl, hdl, tch et ltg). Nous laissons tomber dans le cadre de ce TD le sexe du patient et la variable prog.


#### Référence
Tibshirani, R. J., Johnstone, I., Hastie, T., & Efron, B. (2004). Least angle regression. *The Annals of Statistics*, 32(2), 407–499. DOI:10.1214/009053604000000067


In [ ]:
using CSV 
using DataFrames
using Distributions
using Gadfly
using LinearAlgebra
using Statistics
import Random
import StatsBase

## Chargement des données

In [ ]:
data = CSV.read("diabetes.csv", DataFrame)
select!(data, Not([:prog, :sex]))
first(data,5)

## Création de l'ensemble d'entraînement et de validation

In [ ]:
round(Int, .8*nrow(data))

In [ ]:
Random.seed!(3302)

ntrain = round(Int, .8*nrow(data)) # taille de l'ensemble d'entrainement (80% des données)

train_id = sample(1:nrow(data), ntrain, replace=false, ordered=true)  # on choisit aléatoirement un ss-ens. parmi les données
valid_id = setdiff(1:nrow(data), train_id)  # idem pour l'ens de validation -- c'est le reste des données

train = data[train_id, :]
valid = data[valid_id, :]

## Estimation des paramètres de standardisation des variables

Paramètres estimés avec l'ensemble d'entraînement.

In [ ]:
# Standardisation des variables
Mtrain = Matrix{Float64}(train) # on crée une matrice avec l'ensemble des données d'entrainement

# fit(ZScoreTransform, ...), puis transform(dt, ...) nous permet de standardiser les données selon la colonne 
# (grâce à dims=1. dims=2 --> standardisé par ligne)
dt = StatsBase.fit(StatsBase.ZScoreTransform, Mtrain, dims=1)
transdata = StatsBase.transform(dt, Matrix{Float64}(train))

X = ... # covariables
y = ... # variable d'intérêt

n = length(y)
m = size(X,2)

# Exercice 1 : Régression Ridge

### (a) Estimez les coefficients de régression Ridge en utilisant $\lambda = 0$.

In [ ]:
# section 7.3.3, p.11
λ = 0.
β̂₀ = ...

### (b) Estimez les coefficients de régression Ridge en utilisant $\lambda = 1$.

In [ ]:
λ = 1.
β̂₁ = ...

### (c) Pour $0 \leq \lambda \leq 10$, estimez les coefficients de régression puis affichez-les en fonction de $\lambda$.

Ce graphique s'appelle la trace des coefficients de régression. Lorsqu'il y a multicolinéarité, les estimations des coefficients de régression varient beaucoup en fonction de $\lambda$.

Je vous propose de remplir la matrice $\hat\beta$ constituée de $p$ colonnes. Chaque ligne correspondra aux estimations des coefficients de régression pour une valeur de $\lambda$ donnée. Cette matrice sera par la suite transformée en DataFrame pour un affichage simplifié.

In [ ]:
# on crée un DataFrame vide dans lequel on va stocker les résultats
df = DataFrame(λ = Float64[], paramètre = String[], estimation = Float64[])
coef = ["β₁", "β₂", "β₃", "β₄", "β₅", "β₆", "β₇", "β₈"]

for λ=0:0.01:10
    β̂ = ...
    for i=1:m
        push!(df, [λ, coef[i], β̂[i]])
    end
end

In [ ]:
plot(df, x=:λ, y=:estimation, color = :paramètre, Geom.line)

### (d) Estimez λ en minimisant l'erreur de prédiction sur l'échantillon de validation
Voir la section 7.3.5, p.12 des notes

In [ ]:
# Standardisation des variables de validation
transdata = ...

Xv = ...
yv = ...;

In [ ]:
df = DataFrame(λ = Float64[], rmse = Float64[])

for λ=0:0.01:50
    β̂ = ...      # on doit trouver les coefficients avec les données d'entrainement
    ŷ = ...      # mais les estimations avec les données de validation
    rmse = StatsBase.rmsd(ŷ, yv) # calcul de la racine carré de l’erreur quadratique moyenne
    push!(df, [λ, rmse])
end

λ̂ = df.λ[argmin(df.rmse)]

In [ ]:
plot(df, x=:λ, y=:rmse, Geom.line)

### (e) Tracez la loi *a posteriori* marginale du coefficient de régression correspond à la variable *age*.

Est-ce que cette variable possède un pouvoir prédictif significatif ? On veut savoir si 0 se trouve dans la région de crédibilité 95%

Utiliser la commande `LocationScale(μ, sqrt(Σ), TDist(ν))` pour créer un objet de type *loi de Student multidimensionnelle à ν degrés de liberté, de paramètre de localisation μ et de paramètre d'échelle Σ*.

In [ ]:
# on commence par trouver les coefficients de régression avec λ̂ optimal
β̂ = ...
ŷ = ...
e = ...   # erreur
σ̂² = ... # s^2_lambda (eq 7.11)

In [ ]:
Σ = ... # matrice de covariance de la conditionnelle complète pour la moyenne (eq 7.12)

In [ ]:
posterior_marg = ... #beta[1] et Sigma[1,1] parce qu'on s'intéresse à l'âge
# ici, la fonction LocationScale() permet de rajouter un param. de localisation μ et un param. d'échelle Σ
# à la loi TDist(n-m)

In [ ]:
# on trace la distr a posteriori marginale trouvée précédemment
plot(x->pdf(posterior_marg, x), -.2, .4, xintercept=quantile.(posterior_marg, [.025, .975]), # avec les percentiles 2.5 et 97.5
    Geom.vline(style=:dash, color="red"), Guide.xlabel("β₁"), Guide.ylabel("Densité"))       # pour créer la région de crédibilité

Oui : la région de crédibilité ne comporte pas 0

### (f) Implémentez l'échantillonnage de Gibbs pour générer un échantillon de la loi *a posteriori* de la régression ridge.

Utilisez votre estimation λ̂.

In [ ]:
# fonctions marginales
f₁(σ²::Real) = MvNormal(β̂, σ²*Symmetric(inv(X'X + λ̂*I)))
f₂(β::Vector{<:Real}) = InverseGamma((n+m)/2, (e'e + λ̂*β'β)/2)

it = 10000

# état initial des paramètres
β = Array{Float64}(undef, 8, it)  
σ² = Array{Float64}(undef, it)
β[:,1] = zeros(8)
σ²[1] = 0.005;

# échantillonnage de Gibbs
for i in 2:it
    β[:, i] = rand(f₁(σ²[i-1]))
    σ²[i] = rand(f₂(β[:, i]))
end 

In [ ]:
# trace de β₁
fig = plot(y=β[1,:], Geom.line, Guide.xlabel("Itération"), Guide.ylabel("β₁"))

In [ ]:
# trace de σ²
fig = plot(y=σ²[:], Geom.line, Guide.xlabel("Itération"), Guide.ylabel("σ²"))

In [ ]:
# on coupe la phase de burn-in
β = β[:, 500:end]
σ² = σ²[500:end];

In [ ]:
# superposition de la densité à postériori à l'histogramme des betas1 généré par Gibbs
plot(
    layer(x->pdf(posterior_marg,x), -.5, .5, Theme(default_color=colorant"red"),  # le layer pour la densité
            # ↑ évalue la densité de probabilité de pd, entre -.5 et .5
    xintercept = quantile.(posterior_marg, [.025, .975]), Geom.vline(style=:dash, color="red")),  # les lignes pointillés (intervalles)
    Guide.xlabel("β₁"), Guide.ylabel("Densité"),
    
    layer(x=β[1,:], Geom.histogram(density=true, bincount=30)) # le layer pour l'histogramme (densité=true pour l'échelle)
) 

## Exercice 2 : Exploration stochastique de l'espace des modèles [OPTIONEL]

Dans cet exercice, on reprend le jeux de données *bodyfat.csv* comportant le pourcentage de gras de 20 patientes en fonction des variables explicatives suivantes :

$x_1$ : l'épaisseur des plis de la peau des triceps (en mm) ;<br/>
$x_2$ : le tour de cuisse (en mm) ;<br/>
$x_3$ : la circonférence du bras en (mm).<br/>

Nous allons implémenter l'échantillonnage de Gibbs pour explorer de façon stochastique l'espace des modèele. Puisqu'il n'y a que 3 variables explicatives dans ce cas-ci, il est possible de déterminer quel est le *vrai* meilleur modèle par énumération complète de tous les modèles.


In [ ]:
# Chargement des données
data = CSV.read("bodyfat.csv", DataFrame)
first(data, 5)

In [ ]:
# Standardisation des variables
dt = StatsBase.fit(StatsBase.ZScoreTransform, Matrix{Float64}(data), dims=1)
transdata = StatsBase.transform(dt, Matrix{Float64}(data))

X = transdata[:,1:3] # covariables
y = transdata[:,4]   # variable d'intérêt

n = length(y)
m = size(X,2)

In [ ]:
# Énumération de tous les modèles possibles
itr = reverse.(Iterators.product(fill(0:1,m)...))[:]

M = BitArray[]
for i in eachindex(itr)
   push!(M, [itr[i]...] .== 1)
end
M

### (a) Écrivez une fonction permettant de calculer le BIC d'un modèle.

La fonction devrait prendre comme argument un vecteur de booléens indicant quelles variables sont incluses dans le modèle. La fonction devrait prendre en argument par exemple le vecteur [true true true].

In [ ]:
function getbic(γ::BitArray)
    
    if any(γ)
    
        Xᵧ = X[:,γ]

        β̂ = (Xᵧ'Xᵧ)\Xᵧ'y

        μ̂ = Xᵧ*β̂

        e = y-μ̂

        σ̂² = dot(e,e)/(n-m) 

        BIC = sum(logpdf.(Normal.(μ̂,sqrt(σ̂²)),y)) - (count(γ)+1)/2*log(n)
        
    else
        
        σ̂² = dot(y,y)/(n-m)
        
        BIC = sum(logpdf.(Normal.(0,sqrt(σ̂²)),y)) - (count(γ)+1)/2*log(n)
        
    end
    
    return BIC
    
end

### (b) Calculer le BIC pour chacun des modèles possibles

Quel est le meilleur modèle ?

In [ ]:
BIC = getbic.(M)

In [ ]:
_, ind = findmax(BIC)
M[ind] # modèle 3

### (c) Exécuter la cellule suivante pour parcourir l'espace des modèles avec l'échantillonnage de Gibbs

In [ ]:
function softmax(x::AbstractVector{<:Real})
    
    z = x .- maximum(x)
    
    e = exp.(z)
    
    s = e ./ sum(e)
    
end

In [ ]:
# section 7.4.3, voir l'algo p.16
niter = 10000

γ = trues(3,niter)

for i=2:niter
   γ[:,i] = γ[:,i-1]
    for j=1:m
        
        η₀ = γ[:,i]
        η₁ = γ[:,i]
        η₀[j] = false
        η₁[j] = true
        
        #θ = exp(getbic(η₁)) / ( exp(getbic(η₀)) + exp(getbic(η₁)))
        s = softmax(getbic.([η₀, η₁])) # évite l'instabilité numérique
        θ = s[2]
        
        γ[j,i] = rand(Bernoulli(θ))
        
    end
end

### (d) Exécuter les cellules suivantes pour déterminer le modèle choisi pour chacune des itérations

In [ ]:
function modelid(M::Vector{BitArray},γ::BitArray)
    
    @assert length(γ)==length(M[1])
    
    ind = 1
    
    while M[ind] != γ
        ind +=1
    end
    
    return ind
        
end

In [ ]:
id = [modelid(M,γ[:,i]) for i=1:size(γ,2)]

df = DataFrame(Itération = 1:niter, Modèle = id)

groupby(df, :Modèle)

nb = combine(groupby(df, :Modèle), :Itération=> length => :Itérations)

sort!(nb, :Itérations, rev=true)

In [ ]:
nb

### (e) Selon les résultats en (d), quel est le meilleur modèle ? Est-ce cohérent avec le BIC calculé en (a) ?

In [ ]:
plot(df, x=:Modèle, y=:Itération, Geom.histogram)